# COMP — Comparaison cohérence SIREN ↔ SIRET (vue A : par EJ et ses EG)

Produit un fichier Excel à 4 feuilles montrant la cohérence entre les
deux pipelines, organisé par couple (EJ, EG).

**Source des données** : les 3 fichiers de phase de chaque pipeline (sans passer
par la consolidation finale). La fusion se fait à la volée :
- Phase 1 : toutes les lignes
- Phase 2 (rang 1) : ceux non validés en P1
- Phase 3 (rang 1) : ceux non validés en P1 ni P2

**Logique de cohérence** : pour chaque EG validé, on joint son EJ parent (via
`nmfinessej_stru`) avec le résultat de la sirenisation, puis on compare
`SIRET_EG[:9]` avec le `SIREN_EJ` retenu.

**Statuts produits** :
- **COHERENT** : EG validé + EJ validé + SIRET_EG[:9] == SIREN_EJ
- **INCOHERENT** : EG validé + EJ validé + SIRET_EG[:9] ≠ SIREN_EJ
- **PARTIEL_EG** : EG validé mais EJ parent pas validé
- **PARTIEL_EJ** : EJ validé mais aucun EG validé associé
- **ORPHELIN** : EG validé sans `nmfinessej_stru` renseigné

**Excel produit** : Synthese / Coherent / Incoherent / Partiel

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('../..'))

import pandas as pd
from src.comparaison  import (
    charger_siretisation_depuis_phases, charger_sirenisation_depuis_phases,
    construire_vue_a, synthese_globale,
)
from src.excel_export import export_comparaison_excel
from src.display      import afficher_tableau
from config.settings  import (
    ST_PHASE1, ST_PHASE2, ST_PHASE3,
    SN_PHASE1, SN_PHASE2, SN_PHASE3,
    COMP_GLOBALE, RESULTS_COMP_DIR,
)

RESULTS_COMP_DIR.mkdir(parents=True, exist_ok=True)

## 1. Chargement et fusion des 3 fichiers de phase

In [2]:
df_st = charger_siretisation_depuis_phases(ST_PHASE1, ST_PHASE2, ST_PHASE3)
df_sn = charger_sirenisation_depuis_phases(SN_PHASE1, SN_PHASE2, SN_PHASE3)

print(f'Siretisation (fusion P1 + P2 + P3) : {len(df_st):,} lignes')
print(df_st["statut"].value_counts().to_string())
print(f'\nSirenisation (fusion P1 + P2 + P3) : {len(df_sn):,} lignes')
print(df_sn["statut"].value_counts().to_string())

Siretisation (fusion P1 + P2 + P3) : 161,913 lignes
statut
VALIDE           62899
VALIDE_FORT      31665
DOUTEUX          30841
SANS_SIRET       13124
SIRET_INCONNU    12357
REJETE           10805
SANS_CANDIDAT      222

Sirenisation (fusion P1 + P2 + P3) : 72,651 lignes
statut
VALIDE           26321
VALIDE_FORT      25946
DOUTEUX          11367
SIREN_INCONNU     4103
REJETE            2805
SANS_SIREN        2100
SANS_CANDIDAT        9


In [3]:
import re

def _norm(v):
    if v is None or (isinstance(v, float) and pd.isna(v)):
        return ""
    return re.sub(r"\s", "", str(v).strip())

VALIDES = {"VALIDE_FORT", "VALIDE"}
df_eg_v = df_st[df_st['statut'].isin(VALIDES)]
df_ej_v = df_sn[df_sn['statut'].isin(VALIDES)]

ids_eg_pointent_vers = set(df_eg_v['nmfinessej_stru'].dropna().apply(_norm))
ids_eg_pointent_vers.discard("")
ids_ej_dispos = set(df_ej_v['nmfinessej_stru'].dropna().apply(_norm))
ids_ej_dispos.discard("")

print(f"nmfinessej_stru distincts côté EG (df_st) : {len(ids_eg_pointent_vers):,}")
print(f"nmfinessej_stru distincts côté EJ (df_sn) : {len(ids_ej_dispos):,}")
print(f"Intersection                              : {len(ids_eg_pointent_vers & ids_ej_dispos):,}")
print()
print("Échantillon côté EG :", list(ids_eg_pointent_vers)[:5])
print("Échantillon côté EJ :", list(ids_ej_dispos)[:5])

nmfinessej_stru distincts côté EG (df_st) : 47,510
nmfinessej_stru distincts côté EJ (df_sn) : 52,267
Intersection                              : 46,601

Échantillon côté EG : ['330046954', '380028100', '840015739', '730005865', '760034546']
Échantillon côté EJ : ['330046954', '380028100', '840015739', '730005865', '760034546']


## 2. Synthèse globale Siretisation × Sirenisation

In [4]:
df_synth = synthese_globale(df_st, df_sn)
afficher_tableau(df_synth, 'Synthèse globale Siretisation × Sirenisation', max_lignes=20)

Indicateur,Siretisation,Sirenisation
VALIDE_FORT (Phase 1),25856,23059
VALIDE_FORT (Phase 2),5808,2884
VALIDE_FORT (Phase 3),1,3
VALIDE (Phase 1),34402,15551
VALIDE (Phase 2),26178,9800
VALIDE (Phase 3),2319,970
—,,
DOUTEUX,30841,11367
REJETE,10805,2805
SANS_CANDIDAT,222,9


## 3. Construction de la vue A (couples EG, EJ)

In [5]:
df_vue_a = construire_vue_a(df_st, df_sn)
print(f'Lignes totales : {len(df_vue_a):,}')
print()
print(df_vue_a['statut_coherence'].value_counts().to_string())

Lignes totales : 100,230

statut_coherence
COHERENT      79035
INCOHERENT    14245
PARTIEL_EJ     5666
PARTIEL_EG     1284


## 3. Aperçu des couples cohérents

In [6]:
df_coh = df_vue_a[df_vue_a['statut_coherence'] == 'COHERENT']
afficher_tableau(
    df_coh[['nmfinessej_ej', 'nom_ej', 'siren_ej_retenu',
            'nmfinessej_eg', 'nom_eg', 'siret_eg_retenu', 'siren_du_siret_eg']],
    f'Couples cohérents ({len(df_coh):,} lignes)', max_lignes=10,
)

nmfinessej_ej,nom_ej,siren_ej_retenu,nmfinessej_eg,nom_eg,siret_eg_retenu,siren_du_siret_eg
070007984,SISA DES HAUTES VALLEES D'ARDECHE,879580538,070007984,MAISON DE SANTE HTES VALLEES D'ARDECHE,87958053800012,879580538
090004243,AAD 09,804442119,090004243,SAAD AAD 09,80444211900025,804442119
090004268,APM,531004992,090004268,DOMALIANCE FOIX,53100499200040,531004992
090004300,ASSOCIATION BLEU PRINTEMPS,391482965,090004300,SAAD ASSOCIATION BLEU PRINTEMPS,39148296500053,391482965
090004359,A DEUX MAINS,879754331,090004359,SAAD A DEUX MAINS,87975433100020,879754331
140032582,GEP SANTÉ NORMANDIE,839990397,140032582,GEP SANTÉ NORMANDIE,83999039700011,839990397
170025514,OVELIA 17,821789211,170025514,SAAD OVELIA 17,82178921100025,821789211
190011809,IMAGERIE MEDICALE JM DUCLOUX,440870590,190011809,IMAGERIE MEDICALE JM DUCLOUX,44087059000021,440870590
280007915,SARL AID'ATTITUDE SERVICES,823505250,280007915,SAAD AID'ATTITUDE SERVICES,82350525000030,823505250
280007931,SARL AIDE AU SOURIRE,502412992,280007931,SAAD ADHAP SERVICES,50241299200039,502412992


## 4. Aperçu des incohérences (à arbitrer)

In [7]:
df_coh = df_vue_a[df_vue_a['statut_coherence'] == 'COHERENT']
afficher_tableau(
    df_coh[['nmfinessej_ej', 'nom_ej', 'siren_ej_retenu',
            'nmfinessej_eg', 'nom_eg', 'siret_eg_retenu', 'siren_du_siret_eg']],
    f'Couples cohérents ({len(df_coh):,} lignes)', max_lignes=10,
)

nmfinessej_ej,nom_ej,siren_ej_retenu,nmfinessej_eg,nom_eg,siret_eg_retenu,siren_du_siret_eg
070007984,SISA DES HAUTES VALLEES D'ARDECHE,879580538,070007984,MAISON DE SANTE HTES VALLEES D'ARDECHE,87958053800012,879580538
090004243,AAD 09,804442119,090004243,SAAD AAD 09,80444211900025,804442119
090004268,APM,531004992,090004268,DOMALIANCE FOIX,53100499200040,531004992
090004300,ASSOCIATION BLEU PRINTEMPS,391482965,090004300,SAAD ASSOCIATION BLEU PRINTEMPS,39148296500053,391482965
090004359,A DEUX MAINS,879754331,090004359,SAAD A DEUX MAINS,87975433100020,879754331
140032582,GEP SANTÉ NORMANDIE,839990397,140032582,GEP SANTÉ NORMANDIE,83999039700011,839990397
170025514,OVELIA 17,821789211,170025514,SAAD OVELIA 17,82178921100025,821789211
190011809,IMAGERIE MEDICALE JM DUCLOUX,440870590,190011809,IMAGERIE MEDICALE JM DUCLOUX,44087059000021,440870590
280007915,SARL AID'ATTITUDE SERVICES,823505250,280007915,SAAD AID'ATTITUDE SERVICES,82350525000030,823505250
280007931,SARL AIDE AU SOURIRE,502412992,280007931,SAAD ADHAP SERVICES,50241299200039,502412992


## 5. Aperçu des incohérences (à arbitrer)

In [8]:
df_incoh = df_vue_a[df_vue_a['statut_coherence'] == 'INCOHERENT']
if len(df_incoh) > 0:
    afficher_tableau(
        df_incoh[['nmfinessej_ej', 'nom_ej', 'siren_ej_retenu',
                  'nmfinessej_eg', 'nom_eg', 'siret_eg_retenu', 'siren_du_siret_eg']],
        f'Couples incohérents ({len(df_incoh):,} lignes)', max_lignes=10,
    )
else:
    print('Aucun cas incohérent.')

nmfinessej_ej,nom_ej,siren_ej_retenu,nmfinessej_eg,nom_eg,siret_eg_retenu,siren_du_siret_eg
310031166,SISA MSP MINIMES LES RAISINS,879743722,310031166,MSP MINIMES LES RAISINS,84755111600019,847551116
520004870,CIAS DE L'AGGLOMERATION DE CHAUMONT,265200378,520004870,RESIDENCE AUTONOMIE JACQUES WEIL,20002876900038,200028769
750054157,OPPELIA,382656981,750054157,CHU OPPELIA,32602117700059,326021177
910806728,MAIRIE STE GENEVIEVE DES BOIS,219105491,910806728,RESIDENCE AUTONOMIE ALBERT PERRISSIN,26910101000057,269101010
920170057,ASSOCIATION PAIDEIA,942456385,920170057,CMPP - BAPU INSTITUT EDOUARD CLAPAREDE,34884772400013,348847724
750810806,ETABLIS REGIONAL ENSEIGNEMENT ADAPTE,197532567,750810806,LYCEE RENÉ AUFFRAY,19922149000022,199221490
750000689,CONSEIL REGIONAL D'ILE DE FRANCE,341577245,750000689,ECOLE D'AIDES SOIGNANTS,23750007900213,237500079
750000689,CONSEIL REGIONAL D'ILE DE FRANCE,341577245,750000689,ECOLE D'AUXILIAIRE DE PUERICULTURE,23750007900247,237500079
750000689,CONSEIL REGIONAL D'ILE DE FRANCE,341577245,750000689,ECOLE DE FORMATION PARAMEDICALE,23750007900221,237500079
930812870,COMMUNE D'AULNAY SOUS BOIS,269300026,930812870,RESIDENCE AUTONOMIE LES TAMARIS,21930005000875,219300050


## 5. Export Excel (4 feuilles : Synthese / Coherent / Incoherent / Partiel)

In [9]:
compteurs = export_comparaison_excel(df_vue_a, COMP_GLOBALE)

print('Synthèse de la cohérence :')
print(f'  EJ avec tous EG cohérents      : {compteurs["EJ_tous_coherents"]:,}')
print(f'  EJ avec au moins un incohérent : {compteurs["EJ_avec_incoherent"]:,}')
print(f'  EJ validés sans EG validé      : {compteurs["PARTIEL_EJ"]:,}')
print(f'  EG validés sans EJ validé      : {compteurs["PARTIEL_EG"]:,}')
print(f'  EG orphelins (sans EJ parent)  : {compteurs["ORPHELIN"]:,}')
print()
print(f'  Total couples cohérents   : {compteurs["COHERENT_couples"]:,}')
print(f'  Total couples incohérents : {compteurs["INCOHERENT_couples"]:,}')
print(f'\nFichier : {COMP_GLOBALE}')

Synthèse de la cohérence :
  EJ avec tous EG cohérents      : 40,388
  EJ avec au moins un incohérent : 6,213
  EJ validés sans EG validé      : 5,666
  EG validés sans EJ validé      : 1,284
  EG orphelins (sans EJ parent)  : 0

  Total couples cohérents   : 79,035
  Total couples incohérents : 14,245

Fichier : /home/jovyan/work/projet_finess_sirene/results/comparaison/coherence_globale.xlsx


## Séparation de comparaison par phase

#### Imports + chargement

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../..'))

import pandas as pd
from pathlib import Path
from openpyxl.styles import PatternFill, Font, Alignment

from config.settings import RESULTS_COMP_DIR

FICHIER_SOURCE = RESULTS_COMP_DIR / "coherence_globale.xlsx"
FICHIER_SORTIE = RESULTS_COMP_DIR / "coherence_par_phase.xlsx"

assert FICHIER_SOURCE.exists(), f"Fichier introuvable : {FICHIER_SOURCE}"

# Lecture des feuilles du fichier source
sheets = pd.read_excel(FICHIER_SOURCE, sheet_name=None, dtype=str)
print(f"Feuilles disponibles : {list(sheets.keys())}")

# Feuille Coherent
df_coh = sheets.get("Coherent", pd.DataFrame())
# Feuille Incoherent
df_inc = sheets.get("Incoherent", pd.DataFrame())
# Feuille Partiel (contient PARTIEL_EJ + PARTIEL_EG + ORPHELIN)
df_par = sheets.get("Partiel", pd.DataFrame())

print(f"\nCoherent  : {len(df_coh):,} lignes")
print(f"Incoherent: {len(df_inc):,} lignes")
print(f"Partiel   : {len(df_par):,} lignes")
print()

# Aperçu des colonnes de phase disponibles
for col in ["phase_ej", "phase_eg", "cote_valide"]:
    if col in df_coh.columns:
        print(f"  {col} dans Coherent  : {df_coh[col].value_counts().to_dict()}")
    if col in df_par.columns:
        print(f"  {col} dans Partiel   : {df_par[col].value_counts().to_dict()}")

#### Découpage + export

In [ ]:
from openpyxl.styles import PatternFill, Font, Alignment
from openpyxl.utils import get_column_letter

def style_entete(ws, couleur_hex):
    for cell in ws[1]:
        cell.fill      = PatternFill("solid", start_color=couleur_hex, end_color=couleur_hex)
        cell.font      = Font(bold=True, color="FFFFFF", name="Arial", size=10)
        cell.alignment = Alignment(horizontal="center", vertical="center")
    ws.freeze_panes = "A2"

def auto_width(ws, max_w=50):
    for col in ws.columns:
        w = max(len(str(c.value or "")) for c in col)
        ws.column_dimensions[col[0].column_letter].width = min(w + 3, max_w)

# ─── Découpage Coherent par phase ────────────────────────────────────────────
def to_int(val):
    try:
        return int(float(val))
    except (ValueError, TypeError):
        return None

if not df_coh.empty and "phase_ej" in df_coh.columns and "phase_eg" in df_coh.columns:
    df_coh = df_coh.copy()
    df_coh["_ph_ej"] = df_coh["phase_ej"].apply(to_int)
    df_coh["_ph_eg"] = df_coh["phase_eg"].apply(to_int)

    df_p1     = df_coh[(df_coh["_ph_ej"] == 1) & (df_coh["_ph_eg"] == 1)].drop(columns=["_ph_ej","_ph_eg"])
    df_p2     = df_coh[(df_coh["_ph_ej"] == 2) & (df_coh["_ph_eg"] == 2)].drop(columns=["_ph_ej","_ph_eg"])
    df_p3     = df_coh[(df_coh["_ph_ej"] == 3) & (df_coh["_ph_eg"] == 3)].drop(columns=["_ph_ej","_ph_eg"])
    df_autres = df_coh[
        ~(
            ((df_coh["_ph_ej"]==1)&(df_coh["_ph_eg"]==1)) |
            ((df_coh["_ph_ej"]==2)&(df_coh["_ph_eg"]==2)) |
            ((df_coh["_ph_ej"]==3)&(df_coh["_ph_eg"]==3))
        )
    ].drop(columns=["_ph_ej","_ph_eg"])
else:
    print("⚠ Colonnes phase_ej / phase_eg absentes")
    df_p1 = df_p2 = df_p3 = df_autres = pd.DataFrame()

# ─── Découpage Partiel EJ / EG ───────────────────────────────────────────────
if not df_par.empty and "cote_valide" in df_par.columns:
    df_par_ej = df_par[df_par["cote_valide"].str.contains("EJ", case=False, na=False)]
    df_par_eg = df_par[df_par["cote_valide"].str.contains("EG", case=False, na=False)]
else:
    df_par_ej = df_par_eg = pd.DataFrame()

# ─── Colonne EJ pour les uniques ─────────────────────────────────────────────
COL_EJ = next((c for c in ["id_ej","nmfinessej_ej","nmfinessej_stru"]
               if c in df_p1.columns), None)
print(f"Colonne EJ détectée : {COL_EJ}")

def count_ej(df):
    if COL_EJ is None or df.empty or COL_EJ not in df.columns:
        return 0
    return df[COL_EJ].dropna().nunique()

df_all_coh = pd.concat([df_p1, df_p2, df_p3, df_autres], ignore_index=True)

# ─── Données des deux tableaux ───────────────────────────────────────────────
# Tableau 1 : couples par feuille
t1 = [
    ("Cohérent — Phase 1  (EJ P1 + EG P1)",    len(df_p1),     "1A7341"),
    ("Cohérent — Phase 2  (EJ P2 + EG P2)",    len(df_p2),     "2E7D32"),
    ("Cohérent — Phase 3  (EJ P3 + EG P3)",    len(df_p3),     "558B2F"),
    ("Cohérent — Autres   (phases mixtes)",     len(df_autres), "F9A825"),
    (None, None, None),
    ("Incohérent",                              len(df_inc),    "C00060"),
    (None, None, None),
    ("Partiel EJ  (EJ seul validé)",            len(df_par_ej), "D4A017"),
    ("Partiel EG  (EG seul validé / orphelin)", len(df_par_eg), "E65100"),
    (None, None, None),
    ("TOTAL COHÉRENTS",
     len(df_p1)+len(df_p2)+len(df_p3)+len(df_autres), "TOTAL"),
]

# Tableau 2 : EJ uniques par catégorie
t2 = [
    ("EJ cohérents en Phase 1",           count_ej(df_p1),     "1A7341"),
    ("EJ cohérents en Phase 2",           count_ej(df_p2),     "2E7D32"),
    ("EJ cohérents en Phase 3",           count_ej(df_p3),     "558B2F"),
    ("EJ cohérents (phases mixtes)",      count_ej(df_autres), "F9A825"),
    (None, None, None),
    ("TOTAL EJ cohérents (uniques)",      count_ej(df_all_coh),"TOTAL"),
    (None, None, None),
    ("EJ incohérents (≥ 1 EG incohérent)",count_ej(df_inc),    "C00060"),
    (None, None, None),
    ("EJ partiels (sans EG validé)",      len(df_par_ej),      "D4A017"),
    ("EG partiels (sans EJ validé)",      len(df_par_eg),      "E65100"),
]

# ─── Export ──────────────────────────────────────────────────────────────────
COULEURS_FEUILLES = {
    "Coherent_P1":     "1A7341",
    "Coherent_P2":     "2E7D32",
    "Coherent_P3":     "558B2F",
    "Coherent_Autres": "F9A825",
    "Incoherent":      "C00060",
    "Partiel_EJ":      "D4A017",
    "Partiel_EG":      "E65100",
}

feuilles = [
    ("Coherent_P1",     df_p1),
    ("Coherent_P2",     df_p2),
    ("Coherent_P3",     df_p3),
    ("Coherent_Autres", df_autres),
    ("Incoherent",      df_inc),
    ("Partiel_EJ",      df_par_ej),
    ("Partiel_EG",      df_par_eg),
]

# Couleurs de fond des lignes de données (version pâle)
PALE = {
    "1A7341": "E2F0D9", "2E7D32": "E8F5E9", "558B2F": "F1F8E9",
    "F9A825": "FFF9E6", "C00060": "FCE4EC", "D4A017": "FFF8E7",
    "E65100": "FFF3E0", "TOTAL":  "D9E1F2",
}

def ecrire_tableau(ws, data, col_debut, row_debut, titre, col_valeur_label):
    """Écrit un tableau stylisé dans une feuille à une position donnée."""
    cl = get_column_letter(col_debut)
    cv = get_column_letter(col_debut + 1)

    # Titre
    ws[f"{cl}{row_debut}"] = titre
    ws[f"{cl}{row_debut}"].font = Font(bold=True, color="1F3864", name="Arial", size=11)

    # En-tête
    r = row_debut + 1
    ws[f"{cl}{r}"] = "Indicateur"
    ws[f"{cv}{r}"] = col_valeur_label
    for cell in [ws[f"{cl}{r}"], ws[f"{cv}{r}"]]:
        cell.fill      = PatternFill("solid", start_color="1F3864", end_color="1F3864")
        cell.font      = Font(bold=True, color="FFFFFF", name="Arial", size=10)
        cell.alignment = Alignment(horizontal="center", vertical="center")

    # Données
    for indicateur, valeur, couleur in data:
        r += 1
        if indicateur is None:
            ws[f"{cl}{r}"] = "—"
            continue

        cell_ind = ws[f"{cl}{r}"]
        cell_val = ws[f"{cv}{r}"]
        cell_ind.value = indicateur
        cell_val.value = valeur

        fond = PALE.get(couleur, "FFFFFF")
        for cell in [cell_ind, cell_val]:
            cell.fill      = PatternFill("solid", start_color=fond, end_color=fond)
            cell.alignment = Alignment(horizontal="left" if cell.column==col_debut else "center",
                                       vertical="center")
        if couleur == "TOTAL":
            for cell in [cell_ind, cell_val]:
                cell.font = Font(bold=True, name="Arial", size=10)

    # Largeur colonnes
    ws.column_dimensions[cl].width = 38
    ws.column_dimensions[cv].width = 14


with pd.ExcelWriter(FICHIER_SORTIE, engine="openpyxl") as writer:

    # ── Feuille Synthese (en premier) ──
    # On crée la feuille vide via un df vide pour qu'elle existe
    pd.DataFrame().to_excel(writer, sheet_name="Synthese", index=False)
    ws_synth = writer.sheets["Synthese"]

    ecrire_tableau(ws_synth, t1, col_debut=1, row_debut=1,
                   titre="Répartition des couples par feuille",
                   col_valeur_label="Couples (EG)")
    ecrire_tableau(ws_synth, t2, col_debut=4, row_debut=1,
                   titre="Vue par EJ uniques",
                   col_valeur_label="EJ (uniques)")

    # Séparateur visuel entre les deux tableaux (colonne C vide)
    ws_synth.column_dimensions["C"].width = 3

    # ── Autres feuilles ──
    for nom, df in feuilles:
        if df.empty:
            continue
        df.to_excel(writer, sheet_name=nom, index=False)
        ws = writer.sheets[nom]
        style_entete(ws, COULEURS_FEUILLES[nom])
        auto_width(ws)

print(f"\nExport OK → {FICHIER_SORTIE}")
print()
for nom, df in feuilles:
    print(f"  {nom:<22} : {len(df):>6,} lignes")